In [1]:
import os
import torch
import librosa
import numpy as np
import re
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor, pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score

/ihome/lshangguan/zhw227/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# --- 1. 环境与模型初始化 ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
BASE_PATH = "evaluation"
TRANSCRIPTION_LOG = "evaluation_transcriptions.txt"
SAMPLE_RATE = 16000
WINDOW_SEC = 10

# --- 灵敏度调节 ---
SPEECH_THRESHOLD = 0.5 
MIN_RMS = 0.01          

print(f"初始化系统中... 设备: {DEVICE}")

# A. 加载 AST (语音检测)
ast_model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_extractor = AutoFeatureExtractor.from_pretrained(ast_model_name)
ast_model = AutoModelForAudioClassification.from_pretrained(ast_model_name).to(DEVICE)
ast_model.eval()

# B. 加载 Distil-Whisper (文本转录)
whisper_model_id = "distil-whisper/distil-large-v3"
transcribe_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model_id,
    torch_dtype=TORCH_DTYPE,
    device=0 if DEVICE == "cuda" else -1,
    chunk_length_s=30, # 兼容长片段
)

# --- 2. 标准答案配置 ---
files_config = {
    "airport.wav": [0] * 151 + [1] * 200,
    "avenue.wav": [0] * 151,
    "on_the_bus.wav": [0] * 1 + [1] * 400,
    "residential_street.wav": [0] * 201 + [1] * 400,
    "indoor_home.wav": [0] * 401,
    "restaurant.wav": [0] * 251 + [1] * 400
}

def get_ast_prediction(audio_buffer):
    rms = np.sqrt(np.mean(audio_buffer**2))
    if rms < MIN_RMS: return 0, 0.0
    inputs = ast_extractor(audio_buffer, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = ast_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    speech_prob = float(probs[0, [0, 1, 2, 3]].sum())
    return (1 if speech_prob >= SPEECH_THRESHOLD else 0), speech_prob

# --- 3. 执行评估与转录 ---
all_y_true = []
all_y_pred = []
transcription_results = []

print("\n" + "="*60)
print(f"{'处理文件':<25} | {'进度':<10} | {'转录情况'}")
print("="*60)

for filename, ground_truth in files_config.items():
    file_path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(file_path):
        continue
    
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
    
    file_preds = []
    for i in range(len(ground_truth)):
        start_sec = i * WINDOW_SEC
        end_sec = (i + 1) * WINDOW_SEC
        chunk = audio[int(start_sec * SAMPLE_RATE):int(end_sec * SAMPLE_RATE)]
        
        # 1. AST 检测
        pred, conf = get_ast_prediction(chunk)
        file_preds.append(pred)
        
        # 2. 如果检测为语音，进行转录
        if pred == 1:
            # 这里的 chunk 是 numpy 数组，Whisper pipeline 直接支持
            text = transcribe_pipe(chunk)["text"].strip()
            status = "Speech Found"
        else:
            text = "[NON-SPEECH / SILENT]"
            status = "Skipped"
            
        # 记录结果
        timestamp = f"{start_sec:04d}s-{end_sec:04d}s"
        transcription_results.append({
            "file": filename,
            "time": timestamp,
            "pred": "Speech" if pred == 1 else "Non-speech",
            "gt": "Speech" if ground_truth[i] == 1 else "Non-speech",
            "text": text
        })
        
    all_y_true.extend(ground_truth)
    all_y_pred.extend(file_preds)
    print(f"{filename:<25} | 已完成 | {len(ground_truth)} 片段")

# --- 4. 汇总指标与保存文件 ---
with open(TRANSCRIPTION_LOG, "w", encoding="utf-8") as f:
    f.write("=== 语音检测与转录实验日志 ===\n")
    f.write(f"灵敏度阈值: {SPEECH_THRESHOLD} | 能量门限: {MIN_RMS}\n\n")
    f.write(f"{'文件名':<25} | {'时间戳':<12} | {'预测':<10} | {'真值':<10} | {'转录内容'}\n")
    f.write("-" * 100 + "\n")
    for r in transcription_results:
        f.write(f"{r['file']:<25} | {r['time']:<12} | {r['pred']:<10} | {r['gt']:<10} | {r['text']}\n")

# 计算核心指标
acc = accuracy_score(all_y_true, all_y_pred)
pre = precision_score(all_y_true, all_y_pred, zero_division=0)
rec = recall_score(all_y_true, all_y_pred, zero_division=0)
cm  = confusion_matrix(all_y_true, all_y_pred)

print("\n" + "="*45)
print(f"📊 总体准确率: {acc:.4%}")
print(f"🎯 精确率:     {pre:.4%}")
print(f"📢 召回率:     {rec:.4%}")
print("-" * 45)
print(f"📝 详细转录日志已保存至: {TRANSCRIPTION_LOG}")

初始化系统中... 设备: cuda


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 2205.63it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight] 
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



处理文件                      | 进度         | 转录情况
airport.wav               | 已完成 | 351 片段
avenue.wav                | 已完成 | 151 片段
on_the_bus.wav            | 已完成 | 401 片段
residential_street.wav    | 已完成 | 601 片段
indoor_home.wav           | 已完成 | 401 片段
restaurant.wav            | 已完成 | 651 片段

📊 总体准确率: 96.0876%
🎯 精确率:     94.7043%
📢 召回率:     98.3571%
---------------------------------------------
📝 详细转录日志已保存至: evaluation_transcriptions.txt


In [6]:
import os
import torch
import librosa
import numpy as np
import pandas as pd
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor, pipeline

# --- 1. 配置与模型初始化 ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
BASE_PATH = "evaluation"
SAMPLE_RATE = 16000
WINDOW_SEC = 10

# --- AST 灵敏度调节 ---
SPEECH_THRESHOLD = 0.5  # 概率阈值
MIN_RMS = 0.01          # 能量门限

print(f"🚀 系统启动中... 设备: {DEVICE}")

# 加载检测模型 (AST)
ast_model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_extractor = AutoFeatureExtractor.from_pretrained(ast_model_name)
ast_model = AutoModelForAudioClassification.from_pretrained(ast_model_name).to(DEVICE)
ast_model.eval()

# 加载转录模型 (Distil-Whisper)
whisper_model_id = "distil-whisper/distil-large-v3"
transcribe_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model_id,
    torch_dtype=TORCH_DTYPE,
    device=0 if DEVICE == "cuda" else -1
)

# --- 2. 目标文件及处理逻辑配置 ---
# 仅处理这四个能检测到语音的文件
target_files = {
    "airport.wav": 351,             # 总块数
    "on_the_bus.wav": 401,
    "residential_street.wav": 601,
    "restaurant.wav": 651
}

def is_speech(audio_buffer):
    """AST 语音检测逻辑"""
    rms = np.sqrt(np.mean(audio_buffer**2))
    if rms < MIN_RMS: return False
    
    inputs = ast_extractor(audio_buffer, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = ast_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    speech_prob = float(probs[0, [0, 1, 2, 3]].sum())
    return speech_prob >= SPEECH_THRESHOLD

def get_label(filename, start_sec):
    """根据文件名和时间轴自动判定标签 (Label)"""
    # 逻辑：对于 10s 一个的块，如果其起始时间符合区间，则标记为 1
    if  filename == "on_the_bus.wav":
        # 1010s 及之前 (即起始时间 <= 1000s 的块，其结束时间为 1010s)
        return 1 if start_sec <= 1000 else 0

    elif filename == "airport.wav":
        # 2010s 到 3010s (即起始时间在 [2010, 3000] 之间的块)
        return 1 if 1510 <= start_sec <= 2500 else 0
        
    elif filename == "residential_street.wav":
        # 2010s 到 3010s (即起始时间在 [2010, 3000] 之间的块)
        return 1 if 2010 <= start_sec <= 3000 else 0
        
    elif filename == "restaurant.wav":
        # 2510s 到 3510s (即起始时间在 [2510, 3500] 之间的块)
        return 1 if 2510 <= start_sec <= 3500 else 0
    
    return 0

# --- 3. 执行检测、转录与分类保存 ---
print("\n开始批量处理...")

for filename, total_chunks in target_files.items():
    file_path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(file_path):
        print(f"⚠️ 跳过: {filename} (文件未找到)")
        continue
    
    print(f"正在分析: {filename}...")
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
    
    file_results = []
    
    for i in range(total_chunks):
        start_sec = i * WINDOW_SEC
        end_sec = (i + 1) * WINDOW_SEC
        
        # 提取音频块
        chunk = audio[int(start_sec * SAMPLE_RATE) : int(end_sec * SAMPLE_RATE)]
        
        # 判定是否包含语音
        if is_speech(chunk):
            # 执行转录
            text = transcribe_pipe(chunk)["text"].strip()
            
            # 获取标签
            label = get_label(filename, start_sec)
            
            # 记录条目
            file_results.append({
                "Filename": filename,
                "Timestamp": f"{start_sec}s-{end_sec}s",
                "Transcription": text,
                "Label": label
            })

    # 保存为该文件专属的 CSV
    if file_results:
        df = pd.DataFrame(file_results)
        csv_name = filename.replace(".wav", "_results.csv")
        df.to_csv(csv_name, index=False, encoding="utf-8-sig")
        print(f"✅ 已保存 {len(file_results)} 条识别结果至: {csv_name}")
    else:
        print(f"ℹ️ {filename} 未识别到有效语音片段。")

print("\n✨ 全部任务处理完毕！")

🚀 系统启动中... 设备: cuda


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 2210.25it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight] 



开始批量处理...
正在分析: airport.wav...
✅ 已保存 206 条识别结果至: airport_results.csv
正在分析: on_the_bus.wav...
✅ 已保存 379 条识别结果至: on_the_bus_results.csv
正在分析: residential_street.wav...
✅ 已保存 399 条识别结果至: residential_street_results.csv
正在分析: restaurant.wav...
✅ 已保存 470 条识别结果至: restaurant_results.csv

✨ 全部任务处理完毕！
